**Step-1 Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

sns.set_style("whitegrid")

print("Libraries loaded successfully")

Libraries loaded successfully


**Step 2- Create the Data Set**

In [2]:
np.random.seed(42)

N = 100_000

issue_categories = [
    "Network",
    "Software",
    "Hardware",
    "Access",
    "Security",
    "Email",
    "Database",
    "VPN"
]

priorities = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

client_types = [
    "Small",
    "Medium",
    "Enterprise"
]

df = pd.DataFrame({
    "ticket_id": np.arange(1, N + 1),

    "issue_category": np.random.choice(
        issue_categories,
        N,
        p=[0.18, 0.20, 0.15, 0.15, 0.08, 0.10, 0.06, 0.08]
    ),

    "priority": np.random.choice(
        priorities,
        N,
        p=[0.30, 0.40, 0.23, 0.07]
    ),

    "client_type": np.random.choice(
        client_types,
        N,
        p=[0.50, 0.30, 0.20]
    )
})

df.head()

,ticket_id,issue_category,priority,client_type
0,1,Software,Medium,Small
1,2,VPN,Medium,Small
2,3,Security,Medium,Small
3,4,Access,Medium,Small
4,5,Network,Medium,Small


**step 3-Generate Resolution Time**

In [3]:
base_resolution = {
    "Network": 4.5,
    "Software": 3.0,
    "Hardware": 5.0,
    "Access": 1.5,
    "Security": 6.0,
    "Email": 2.0,
    "Database": 5.5,
    "VPN": 3.5
}

priority_multiplier = {
    "Low": 0.8,
    "Medium": 1.0,
    "High": 1.3,
    "Critical": 1.7
}

df["resolution_time"] = df.apply(
    lambda row:
    base_resolution[row["issue_category"]]
    * priority_multiplier[row["priority"]]
    * np.random.lognormal(mean=0, sigma=0.25),
    axis=1
)

df["resolution_time"] = df["resolution_time"].round(2)

df.head()


,ticket_id,issue_category,priority,client_type,resolution_time
0,1,Software,Medium,Small,2.94
1,2,VPN,Medium,Small,2.64
2,3,Security,Medium,Small,5.50
3,4,Access,Medium,Small,1.79
4,5,Network,Medium,Small,3.98


**Step 4- Generate engineer hourly cost**

In [5]:
engineer_cost_by_category = {
    "Network": 55,
    "Software": 45,
    "Hardware": 50,
    "Access": 35,
    "Security": 75,
    "Email": 40,
    "Database": 70,
    "VPN": 50
}

df["cost_per_engineer_hour"] = (
    df["issue_category"]
    .map(engineer_cost_by_category)
)

df["cost_per_engineer_hour"] = (
    df["cost_per_engineer_hour"]
    * np.random.uniform(0.90, 1.10, N)
).round(2)


**Step 5- Generate escalation count**

In [6]:
priority_escalation_probability = {
    "Low": 0.05,
    "Medium": 0.10,
    "High": 0.18,
    "Critical": 0.28
}

category_escalation_probability = {
    "Network": 0.15,
    "Software": 0.10,
    "Hardware": 0.18,
    "Access": 0.05,
    "Security": 0.25,
    "Email": 0.07,
    "Database": 0.22,
    "VPN": 0.14
}

def generate_escalation(row):
    probability = (
        priority_escalation_probability[row["priority"]]
        + category_escalation_probability[row["issue_category"]]
    ) / 2
    
    return np.random.binomial(3, probability)

df["escalation_count"] = df.apply(
    generate_escalation,
    axis=1
)

df["escalation_count"].value_counts().sort_index()


escalation_count
0    67402
1    27824
2     4507
3      267
Name: count, dtype: int64

**Step 6- Generate customer satisfaction**

In [7]:
df["csat"] = (
    4.7
    - (df["resolution_time"] * 0.06)
    - (df["escalation_count"] * 0.20)
    + np.random.normal(0, 0.25, N)
)

df["csat"] = df["csat"].clip(1, 5).round(2)

df["csat"].describe()


count    100000.000000
mean          4.382137
std           0.310365
min           2.740000
25%           4.180000
50%           4.400000
75%           4.600000
max           5.000000
Name: csat, dtype: float64

**Step 7- check dataset**

In [8]:
df.head()


,ticket_id,issue_category,priority,client_type,resolution_time,cost_per_engineer_hour,escalation_count,csat
0,1,Software,Medium,Small,2.94,44.43,0,4.51
1,2,VPN,Medium,Small,2.64,54.34,0,5.00
2,3,Security,Medium,Small,5.50,69.11,1,4.47
3,4,Access,Medium,Small,1.79,33.18,0,4.33
4,5,Network,Medium,Small,3.98,60.43,1,4.43


In [9]:
df.shape


(100000, 8)

In [10]:
df.columns.tolist()


['ticket_id',
 'issue_category',
 'priority',
 'client_type',
 'resolution_time',
 'cost_per_engineer_hour',
 'escalation_count',
 'csat']

**Step 8- Add missing data**

In [12]:
missing_columns = [
    "issue_category",
    "priority",
    "resolution_time",
    "cost_per_engineer_hour",
    "client_type",
    "escalation_count"
]

for col in missing_columns:
    mask = np.random.random(N) < 0.10
    df.loc[mask, col] = np.nan


In [13]:
df.isnull().sum()


ticket_id                     0
issue_category            19103
priority                  18954
client_type               19030
resolution_time           18968
cost_per_engineer_hour    19055
escalation_count          19009
csat                          0
dtype: int64

In [14]:
(df.isnull().sum() / len(df) * 100).round(2)


ticket_id                  0.00
issue_category            19.10
priority                  18.95
client_type               19.03
resolution_time           18.97
cost_per_engineer_hour    19.06
escalation_count          19.01
csat                       0.00
dtype: float64

**Step 9 - Save the dataset**

In [15]:
df.to_csv(
    "IT_support_dataset.csv",
    index=False
)

print("Dataset saved successfully!")


Dataset saved successfully!


**step 10- cleaning data**

In [16]:
df["issue_category"] = df["issue_category"].fillna("Unknown")

df["priority"] = df["priority"].fillna("Unknown")

df["client_type"] = df["client_type"].fillna("Unknown")

df["resolution_time"] = df["resolution_time"].fillna(
    df["resolution_time"].median()
)

df["cost_per_engineer_hour"] = df[
    "cost_per_engineer_hour"
].fillna(
    df["cost_per_engineer_hour"].median()
)

df["escalation_count"] = df[
    "escalation_count"
].fillna(0)


In [17]:

df["csat"] = df["csat"].fillna(
    df["csat"].median()
)


**Step 11-  Calcuate ticket cost**

In [18]:
df["ticket_cost"] = (
    df["resolution_time"]
    * df["cost_per_engineer_hour"]
)

print(
    "Total Support Cost:",
    df["ticket_cost"].sum()
)

print(
    "Average Ticket Cost:",
    df["ticket_cost"].mean()
)


Total Support Cost: 20578908.3953
Average Ticket Cost: 205.78908395300002


**Step 12- Calculate escalation rate**

In [19]:
df["escalated"] = (
    df["escalation_count"] > 0
).astype(int)

escalation_rate = (
    df["escalated"].mean() * 100
)

print(
    "Escalation Rate:",
    round(escalation_rate, 2),
    "%"
)


Escalation Rate: 26.37 %


**Step 13-Calculate CSAT**

In [20]:
average_csat = df["csat"].mean()

print(
    "Average CSAT:",
    round(average_csat, 2)
)

print("Target CSAT: > 4.2")


Average CSAT: 4.38
Target CSAT: > 4.2


**Step 14- Calculate the 20% cost target**

In [21]:
current_cost = df["ticket_cost"].sum()

required_savings = current_cost * 0.20

target_cost = current_cost * 0.80

print("Current Cost:", current_cost)
print("Required 20% Savings:", required_savings)
print("Target Cost:", target_cost)


Current Cost: 20578908.3953
Required 20% Savings: 4115781.6790600005
Target Cost: 16463126.716240002
